# mse-reconstruction-loss — worked example 1: Compute channel-weighted MSE reconstruction loss

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `mse-reconstruction-loss`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`F.mse_loss` with `reduction='none'` returns a per-element tensor of the same shape as the inputs, giving you full control over which axes to reduce and by what weights. For multi-channel images you can weight channels differently — for example, weighting the luminance channel more than chroma — by multiplying the per-element loss by a channel weight tensor before taking the mean.

## Worked solution

**Step 1 — compute per-element MSE.** Call `F.mse_loss(pred, target, reduction='none')` to get a tensor of shape `(B, C, H, W)` where each entry is `(pred - target)²`.

**Step 2 — define per-channel weights.** Create a weight vector of shape `(C,)`, one scalar per channel. Reshape it to `(1, C, 1, 1)` so it broadcasts over the batch, height, and width axes.

**Step 3 — apply weights.** Multiply the per-element MSE tensor by the reshaped weights: `weighted = per_elem * channel_weights`. The result still has shape `(B, C, H, W)` but each channel's errors are scaled.

**Step 4 — reduce to a scalar.** Call `.mean()` on the weighted tensor to get a single scalar loss. This is the weighted reconstruction loss that can be directly passed to `.backward()`.

**Why this is useful.** In RGB autoencoders, the green channel carries the most luminance information for human perception. Upweighting it biases reconstruction toward preserving perceptual brightness.

In [ ]:
import torch
import torch.nn.functional as F

torch.manual_seed(0)

def weighted_channel_mse(pred: torch.Tensor, target: torch.Tensor,
                         channel_weights: torch.Tensor) -> torch.Tensor:
    """
    Compute channel-weighted MSE reconstruction loss.
    pred, target: (B, C, H, W)
    channel_weights: (C,) -- one weight per channel
    Returns: scalar weighted loss.
    """
    per_elem = F.mse_loss(pred, target, reduction='none')  # (B, C, H, W)
    # Reshape weights to broadcast over B, H, W
    w = channel_weights.view(1, -1, 1, 1).float()         # (1, C, 1, 1)
    weighted = per_elem * w                               # (B, C, H, W)
    return weighted.mean()                                # scalar

# Exercise: RGB-like batch, upweight green channel.
torch.manual_seed(11)
B, C, H, W = 4, 3, 16, 16
pred   = torch.randn(B, C, H, W)
target = torch.randn(B, C, H, W)

# Weights: R=0.3, G=0.6, B=0.1
weights = torch.tensor([0.3, 0.6, 0.1])

loss = weighted_channel_mse(pred, target, weights)
print(f"Weighted MSE loss: {loss.item():.6f}")

# Sanity: unit weights should match plain mse_loss
unit_weights = torch.ones(3)
loss_unit = weighted_channel_mse(pred, target, unit_weights)
loss_plain = F.mse_loss(pred, target)
print(f"Unit-weight loss: {loss_unit.item():.6f}, plain MSE: {loss_plain.item():.6f}")
assert abs(loss_unit.item() - loss_plain.item()) < 1e-5, "unit weights should match plain MSE"
print("Sanity check passed!")